# Test Koneksi: Spark + MinIO + Iceberg

## 1. Inisialisasi SparkSession

In [1]:
import os
from pyspark.sql import SparkSession

JARS = ",".join([
    "/home/jovyan/extra-jars/iceberg-spark-runtime-3.5_2.12-1.4.3.jar",
    "/home/jovyan/extra-jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/extra-jars/aws-java-sdk-bundle-1.12.262.jar",
])
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {JARS} pyspark-shell"

spark = (
    SparkSession.builder
    .appName("test-connection")
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://lakehouse/")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("SparkSession created successfully")

Spark version: 3.5.0
SparkSession created successfully


## 2. Test Akses MinIO (S3A)

In [2]:
# Tulis data dummy ke MinIO
df_test = spark.createDataFrame([(1, 'hello'), (2, 'world')], ['id', 'value'])
df_test.write.mode('overwrite').parquet('s3a://lakehouse/test/sample')

# Baca kembali
df_read = spark.read.parquet('s3a://lakehouse/test/sample')
df_read.show()
print('MinIO read/write OK')

+---+-----+
| id|value|
+---+-----+
|  1|hello|
|  2|world|
+---+-----+

MinIO read/write OK


## 3. Test Iceberg Table

In [3]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.test')

spark.sql('''
    CREATE TABLE IF NOT EXISTS local.test.sample (
        id   BIGINT,
        name STRING
    )
    USING iceberg
''')

spark.sql("INSERT INTO local.test.sample VALUES (1, 'Alice'), (2, 'Bob')")

spark.sql('SELECT * FROM local.test.sample').show()
print('Iceberg table OK')

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

Iceberg table OK


## 4. Cleanup

In [4]:
spark.sql('DROP TABLE IF EXISTS local.test.sample')
spark.stop()
print('Done')

Done
